# Edge-Case Review — LLM-Filtered Comments

Picks random comments from the full `reddit_comments.csv`, asks the LLM whether
each is financially relevant, and collects the first **N_TARGET** that are.

**Your job:** open `eval/review_edge_cases.csv` and fill in:
- `corrected_tickers` — the tickers the comment is *actually about* (may differ from what the LLM picked)
- `corrected_sentiment` — if the LLM got it wrong
- `notes` — flag edge cases: sarcasm, indirect mention, multiple companies where only one matters, etc.

Return the filled CSV when done.

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from IPython.display import display, HTML
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from enum import Enum

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_FILE    = "reddit_comments.csv"
EVAL_DIR      = Path("eval")
OUTPUT_FILE   = EVAL_DIR / "review_edge_cases.csv"

N_TARGET      = 20    # how many relevant comments to collect
BATCH_SIZE    = 10    # comments sent to LLM in parallel per round
N_WORKERS     = 8     # parallel LLM threads
SEED          = 42    # change to get a different random draw

MODEL_NAME    = "mistralai/Mistral-Small-3.2-24B-Instruct-2506"
VLLM_ENDPOINT = "http://127.0.0.1:8000/v1"
API_KEY       = "password"

In [ ]:
# ── LLM setup — plain JSON calls (with_structured_output hangs with Mistral/vLLM) ──
import json as _json

_llm = ChatOpenAI(
    base_url=VLLM_ENDPOINT,
    api_key=API_KEY,
    model=MODEL_NAME,
    temperature=0,
    max_retries=3,
    timeout=90,
)

SYSTEM_PROMPT = """You are a financial NLP system that analyzes Reddit comments for stock market signals.

Respond with ONLY a valid JSON object in this exact format:
{"tickers": ["AAPL", "TSLA"], "sentiment": "positive", "is_relevant": true}

Rules:
- is_relevant: true ONLY if at least one publicly traded company is clearly mentioned or implied
- tickers: list of standard US ticker symbols; empty list [] if not relevant
- sentiment: exactly one of: "very positive", "positive", "neutral", "negative", "very negative"
- If not relevant: {"tickers": [], "sentiment": "neutral", "is_relevant": false}
- Output JSON only — no explanation, no markdown, no extra text
"""

VALID_SENTIMENTS = {"very positive", "positive", "neutral", "negative", "very negative"}

def analyze(comment: str) -> dict | None:
    """Return analysis dict, or None on unrecoverable error."""
    try:
        response = _llm.invoke([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": str(comment)[:2000]},
        ])
        text = response.content.strip()
        start = text.find("{")
        end   = text.rfind("}") + 1
        if start == -1 or end == 0:
            print(f"  No JSON in response: {text[:120]}")
            return None
        data = _json.loads(text[start:end])
        sentiment = data.get("sentiment", "neutral").lower().strip()
        if sentiment not in VALID_SENTIMENTS:
            sentiment = "neutral"
        return {
            "tickers":     [t.upper().strip() for t in data.get("tickers", []) if t.strip()],
            "sentiment":   sentiment,
            "is_relevant": bool(data.get("is_relevant", False)),
        }
    except Exception as e:
        print(f"  LLM error: {e}")
        return None

print("LLM client ready.")


In [ ]:
# ── Load source data, exclude already-labeled comments ────────────────────────
def stable_hash(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()[:8]

source = pd.read_csv(INPUT_FILE)
source["id"] = source["comments"].apply(stable_hash)
print(f"Source: {len(source):,} comments")

# Collect IDs already used in calibration + batch files
used_ids = set()
for path in EVAL_DIR.glob("calibration_*.csv"):
    df = pd.read_csv(path)
    if "id" in df.columns:
        used_ids.update(df["id"].dropna().astype(str))
for path in EVAL_DIR.glob("batch_*.csv"):
    df = pd.read_csv(path)
    if "id" in df.columns:
        used_ids.update(df["id"].dropna().astype(str))

pool = source[~source["id"].isin(used_ids)].sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Pool after excluding already-labeled: {len(pool):,} comments")

In [ ]:
# ── Run LLM in batches until N_TARGET relevant comments found ─────────────────
collected = []   # list of dicts: source row + LLM result
cursor    = 0

while len(collected) < N_TARGET and cursor < len(pool):
    batch_rows = pool.iloc[cursor : cursor + BATCH_SIZE]
    cursor += BATCH_SIZE

    futures = {}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for _, row in batch_rows.iterrows():
            futures[ex.submit(analyze, row["comments"])] = row

        for fut in as_completed(futures):
            row    = futures[fut]
            result = fut.result()
            if result is None or not result["is_relevant"]:
                continue
            collected.append({
                "id":          row["id"],
                "datetime":    row["datetime"],
                "subreddits":  row["subreddits"],
                "comments":    row["comments"],
                "llm_tickers":   ",".join(result["tickers"]),
                "llm_sentiment": result["sentiment"],
            })
            if len(collected) >= N_TARGET:
                break

    print(f"  Screened {cursor} comments — {len(collected)}/{N_TARGET} relevant so far")

print(f"\nDone. Collected {len(collected)} relevant comments after screening {cursor} total.")

In [ ]:
# ── Display results ───────────────────────────────────────────────────────────
results_df = pd.DataFrame(collected[:N_TARGET])

def render_table(df):
    rows = []
    for i, row in df.iterrows():
        text = str(row["comments"]).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        rows.append(
            f"<tr>"
            f"<td style='padding:6px;font-weight:bold;vertical-align:top'>{i+1}</td>"
            f"<td style='padding:6px;font-family:monospace;font-size:0.8em;vertical-align:top;color:#555'>{row['id']}</td>"
            f"<td style='padding:6px;vertical-align:top;max-width:500px;white-space:pre-wrap'>{text}</td>"
            f"<td style='padding:6px;vertical-align:top;font-weight:bold;color:#1a7abf'>{row['llm_tickers']}</td>"
            f"<td style='padding:6px;vertical-align:top;color:#666'>{row['llm_sentiment']}</td>"
            f"</tr>"
        )
    header = (
        "<tr style='background:#f0f0f0'>"
        "<th style='padding:6px'>#</th>"
        "<th style='padding:6px'>id</th>"
        "<th style='padding:6px'>comment</th>"
        "<th style='padding:6px'>llm_tickers</th>"
        "<th style='padding:6px'>llm_sentiment</th>"
        "</tr>"
    )
    return HTML(f"<table border='1' cellspacing='0' style='border-collapse:collapse;width:100%'>{header}{''.join(rows)}</table>")

display(render_table(results_df))

In [ ]:
# ── Save review sheet ─────────────────────────────────────────────────────────
review = results_df[["id", "datetime", "subreddits", "comments", "llm_tickers", "llm_sentiment"]].copy()
review["corrected_tickers"]   = ""   # fill in: only tickers the comment is actually about
review["corrected_sentiment"] = ""   # fill in: if the LLM got the sentiment wrong
review["notes"]               = ""   # flag edge cases (sarcasm, indirect mention, etc.)

EVAL_DIR.mkdir(exist_ok=True)
review.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(review)} rows to {OUTPUT_FILE}")
print("Fill in 'corrected_tickers', 'corrected_sentiment', and 'notes', then return the file.")